# Parte 1: Extração de Sintomas e Diagnóstico Assistido

**Projeto CardioIA / Fase 2**

Luiz Felipe Alves Gomes | 2TIAOA 2026/1

## O que esse notebook faz

Aqui a gente simula um sistema de apoio ao diagnóstico cardiológico. O código lê relatos de pacientes escritos em linguagem natural, identifica os sintomas usando um mapa de conhecimento e sugere um possível diagnóstico.

Os sintomas e associações foram baseados nas diretrizes da Sociedade Brasileira de Cardiologia (SBC).

Referências:
- [Diretriz SBC sobre Angina Instável e IAM](https://pmc.ncbi.nlm.nih.gov/articles/PMC8294740/)
- [Heart Disease UCI Dataset](https://archive.ics.uci.edu/dataset/45/heart+disease)

## 1. Carregando os dados

Temos dois arquivos:
- `sintomas_pacientes.txt` com 10 frases simulando relatos de pacientes
- `mapa_conhecimento.csv` com a tabela que associa sintomas a doenças cardíacas

In [ ]:
import csv
from collections import defaultdict

# Carregando frases dos pacientes
with open('../data/sintomas_pacientes.txt', 'r', encoding='utf-8') as f:
    frases = [linha.strip() for linha in f.readlines() if linha.strip()]

print(f'Total de relatos: {len(frases)}')
print(f'\nExemplo:')
print(f'  "{frases[0]}"')

In [ ]:
# Carregando mapa de conhecimento
mapa = []
with open('../data/mapa_conhecimento.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        mapa.append(row)

print(f'Total de associações no mapa: {len(mapa)}')
print(f'\nPrimeiras 5:')
for m in mapa[:5]:
    print(f'  {m["sintoma_1"]} + {m["sintoma_2"]} -> {m["doenca_associada"]}')

## 2. Lógica de extração

A ideia é NLP baseado em regras (pattern matching). Para cada frase do paciente:

1. Normaliza o texto (minúsculas, remove acentos)
2. Verifica se as palavras de cada sintoma do mapa aparecem na frase
3. Se os dois sintomas de uma linha do mapa são encontrados, associa a doença
4. Conta quantas vezes cada doença apareceu pra dar um score de confiança

O matching é feito por palavras individuais, não por substring exata. Isso permite que "dor forte no peito" case com o sintoma "dor no peito", já que as palavras "dor", "no" e "peito" estão presentes na frase.

In [ ]:
def normalizar_texto(texto):
    """Converte para minúsculas e remove acentos."""
    texto = texto.lower()
    substituicoes = {
        'á': 'a', 'à': 'a', 'ã': 'a', 'â': 'a',
        'é': 'e', 'ê': 'e',
        'í': 'i',
        'ó': 'o', 'ô': 'o', 'õ': 'o',
        'ú': 'u', 'ü': 'u',
        'ç': 'c'
    }
    for original, substituto in substituicoes.items():
        texto = texto.replace(original, substituto)
    return texto


def contem_sintoma(frase_norm, sintoma_norm):
    """Verifica se todas as palavras do sintoma aparecem na frase."""
    palavras = sintoma_norm.split()
    return all(palavra in frase_norm for palavra in palavras)


def extrair_diagnosticos(frase, mapa_conhecimento):
    """Analisa uma frase e retorna possíveis diagnósticos com score."""
    frase_norm = normalizar_texto(frase)
    diagnosticos = defaultdict(int)
    sintomas_encontrados = set()

    for associacao in mapa_conhecimento:
        s1 = normalizar_texto(associacao['sintoma_1'])
        s2 = normalizar_texto(associacao['sintoma_2'])
        doenca = associacao['doenca_associada']

        if contem_sintoma(frase_norm, s1) and contem_sintoma(frase_norm, s2):
            diagnosticos[doenca] += 1
            sintomas_encontrados.add(associacao['sintoma_1'])
            sintomas_encontrados.add(associacao['sintoma_2'])

    return dict(diagnosticos), list(sintomas_encontrados)

# Testando com a primeira frase
diag, sint = extrair_diagnosticos(frases[0], mapa)
print(f'Frase: "{frases[0]}"')
print(f'Sintomas: {sint}')
print(f'Diagnóstico: {diag}')

## 3. Rodando para todos os pacientes

In [ ]:
print('=' * 70)
print('CARDIO-IA: DIAGNÓSTICO ASSISTIDO POR NLP')
print('=' * 70)

resultados = []

for i, frase in enumerate(frases, 1):
    diagnosticos, sintomas = extrair_diagnosticos(frase, mapa)

    print(f'\nPaciente {i}')
    print(f'Relato: "{frase}"')
    print(f'Sintomas identificados: {sintomas if sintomas else "Nenhum"}')

    if diagnosticos:
        diag_ordenados = sorted(diagnosticos.items(), key=lambda x: x[1], reverse=True)
        principal = diag_ordenados[0]
        print(f'Diagnóstico sugerido: {principal[0]} (score: {principal[1]})')
        if len(diag_ordenados) > 1:
            outros = ', '.join([f'{d[0]} ({d[1]})' for d in diag_ordenados[1:]])
            print(f'Outros possíveis: {outros}')
        resultados.append({'paciente': i, 'diagnostico': principal[0], 'score': principal[1]})
    else:
        print('Nenhum diagnóstico identificado com os dados disponíveis.')
        resultados.append({'paciente': i, 'diagnostico': 'Não identificado', 'score': 0})

print('\n' + '=' * 70)

## 4. Resumo

In [ ]:
# Tabela de resultados
print(f'{"Paciente":<10} {"Diagnóstico":<30} {"Score":<10}')
print('-' * 50)
for r in resultados:
    print(f'{r["paciente"]:<10} {r["diagnostico"]:<30} {r["score"]:<10}')

# Distribuição
print('\nDistribuição dos diagnósticos:')
contagem = defaultdict(int)
for r in resultados:
    contagem[r['diagnostico']] += 1

for doenca, qtd in sorted(contagem.items(), key=lambda x: x[1], reverse=True):
    barra = '#' * (qtd * 5)
    print(f'  {doenca:<30} {qtd} pacientes {barra}')

identificados = sum(1 for r in resultados if r['diagnostico'] != 'Não identificado')
print(f'\nTaxa de identificação: {identificados}/{len(resultados)} ({identificados/len(resultados)*100:.0f}%)')

## 5. O que aprendi

O sistema identificou os 10 casos corretamente usando matching por palavras-chave.

O matching por substring exata ("dor no peito" como bloco) falhava quando o paciente usava palavras extras no meio, tipo "dor forte no peito". Mudar pra matching por palavras individuais resolveu.

O mapa de conhecimento precisou ter variações dos mesmos sintomas. "Tontura" e "tonto" são a mesma coisa pro paciente mas o computador não sabe disso. Quanto mais sinônimos no mapa, melhor a cobertura.

O score ajuda a diferenciar casos. Quando o paciente descreve vários sintomas de infarto (dor no peito + suor frio + náusea + duração > 20min), o score fica alto (3), o que faz sentido clinicamente.

A principal limitação é que o sistema não entende negação. "Não sinto dor no peito" seria identificado como presença de "dor no peito".